# Customer Churn Prediction
## Notebook 2 — Data Preprocessing

**Goal:** Clean and encode the raw data so it's ready for model training.  
**Output:** `data/churn_clean.csv`

### Steps
1. Fix `TotalCharges` — convert to numeric, fill 11 missing values
2. Drop `customerID` — not useful for prediction
3. Encode binary columns — Yes/No → 1/0
4. One-hot encode multi-category columns
5. Save clean dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Raw shape:', df.shape)

## Step 1 — Fix TotalCharges

`TotalCharges` loaded as `object` instead of `float` because 11 rows contain whitespace `' '`.
These belong to customers with `tenure=0` (just signed up), so their total charges are logically 0.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges dtype:', df['TotalCharges'].dtype)
print('Nulls remaining:', df['TotalCharges'].isnull().sum())

## Step 2 — Drop customerID and Encode Target

In [ ]:
# customerID is a random identifier — no predictive value
df = df.drop(columns=['customerID'])

# Encode target: Yes → 1, No → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Churn distribution:')
print(df['Churn'].value_counts())

## Step 3 — Encode Binary Columns

Columns with only Yes/No values (or equivalents like 'No phone service') are mapped to 1/0.

In [ ]:
# gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# All Yes/No columns
binary_cols = [
    'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in binary_cols:
    df[col] = df[col].map({
        'Yes': 1,
        'No': 0,
        'No phone service': 0,
        'No internet service': 0
    })

# Fill any remaining NaNs with 0
df = df.fillna(0)

print('Nulls after encoding:', df.isnull().sum().sum())

## Step 4 — One-Hot Encode Multi-Category Columns

Columns with 3+ categories can't be mapped to a single number — that would imply a false ordering.
We use one-hot encoding instead: each category becomes its own 0/1 column.

In [ ]:
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'])

print('Final shape:', df.shape)
print('\nAll columns:')
print(df.columns.tolist())
print('\nAll dtypes object?', (df.dtypes == 'object').any())

## Step 5 — Save Clean Dataset

In [ ]:
df.to_csv('../data/churn_clean.csv', index=False)
print('Saved to data/churn_clean.csv')
print('Shape:', df.shape)
df.head(3)